# Week 3 — Data Transformation and Aggregation

This week's work focused on transforming the cleaned dataset into useful analytical variables.

The main tasks included combining category metadata, creating calculated fields, grouping records, and generating summary tables that could be used for exploratory analysis.

In [1]:
import pandas as pd
import json
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/INvideos.csv")

analysis = df.drop_duplicates().copy()

analysis["trending_date_parsed"] = pd.to_datetime(
    analysis["trending_date"],
    format="%y.%d.%m",
    errors="coerce"
)

analysis["publish_datetime"] = pd.to_datetime(
    analysis["publish_time"],
    errors="coerce",
    utc=True
)

In [2]:
with open("../data/IN_category_id.json", encoding="utf-8") as f:
    categories = json.load(f)

category_df = pd.DataFrame([
    {
        "category_id": int(item["id"]),
        "category_name": item["snippet"]["title"]
    }
    for item in categories["items"]
])

analysis = analysis.merge(
    category_df,
    on="category_id",
    how="left"
)

analysis[["category_id", "category_name"]].drop_duplicates().sort_values(
    "category_id"
)

,category_id,category_name
0,1,Film & Animation
172,2,Autos & Vehicles
6,10,Music
920,15,Pets & Animals
135,17,Sports
23,19,Travel & Events
4511,20,Gaming
15,22,People & Blogs
3,23,Comedy
2,24,Entertainment


In [3]:
unmapped = analysis.loc[
    analysis["category_name"].isna(),
    "category_id"
].unique()

print("Unmapped category IDs:", unmapped)

Unmapped category IDs: [29]


In [4]:
analysis["engagement_total"] = (
    analysis["likes"]
    + analysis["dislikes"]
    + analysis["comment_count"]
)

analysis["like_rate"] = np.where(
    analysis["views"] > 0,
    analysis["likes"] / analysis["views"],
    np.nan
)

analysis["comment_rate"] = np.where(
    analysis["views"] > 0,
    analysis["comment_count"] / analysis["views"],
    np.nan
)

analysis["dislike_rate"] = np.where(
    analysis["views"] > 0,
    analysis["dislikes"] / analysis["views"],
    np.nan
)

analysis[
    ["views", "likes", "comment_count",
     "engagement_total", "like_rate", "comment_rate"]
].head()

,views,likes,comment_count,engagement_total,like_rate,comment_rate
0,1096327,33966,882,35646,0.030982,0.000805
1,590101,735,0,1639,0.001246,0.000000
2,473988,2011,149,2403,0.004243,0.000314
3,1242680,70353,2684,74661,0.056614,0.002160
4,464015,492,66,851,0.001060,0.000142


In [5]:
analysis["title_length"] = analysis["title"].fillna("").str.len()

analysis[["title", "title_length"]].head(10)

,title,title_length
0,Sharry Mann: Cute Munda ( Song Teaser) | Parmi...,81
1,"पीरियड्स के समय, पेट पर पति करता ऐसा, देखकर दं...",58
2,Stylish Star Allu Arjun @ ChaySam Wedding Rece...,58
3,Eruma Saani | Tamil vs English,30
4,why Samantha became EMOTIONAL @ Samantha naga ...,88
5,"MCA (Middle Class Abbayi) TEASER - Nani,Sai Pa...",91
6,Daang ( Full Video ) | Mankirt Aulakh | Sukh S...,96
7,Padmavati : Ek Dil Ek Jaan Video Song | Deepik...,96
8,Chiranjeevi in Naga Chaitanya - Samantha Recep...,97
9,New bike vs Old bike - the reality,34


In [6]:
category_summary = (
    analysis.groupby("category_name", dropna=False)
    .agg(
        videos=("video_id", "count"),
        median_views=("views", "median"),
        median_likes=("likes", "median"),
        median_comments=("comment_count", "median"),
        median_like_rate=("like_rate", "median")
    )
    .sort_values("videos", ascending=False)
)

category_summary

,videos,median_views,median_likes,median_comments,median_like_rate
category_name,,,,,
Entertainment,14764,250234.0,1802.0,195.0,0.006351
News & Politics,4709,174239.0,1066.0,154.0,0.005706
Music,3292,825655.0,15924.5,948.0,0.021255
Comedy,2967,448912.0,17668.0,1354.0,0.050885
People & Blogs,2367,234387.0,1402.0,154.0,0.005986
Film & Animation,1463,657768.0,7716.0,557.0,0.016362
Education,1166,50872.0,2448.5,269.5,0.048870
Howto & Style,801,245371.0,3123.0,408.0,0.010329
Sports,646,655190.0,6075.0,559.0,0.009895


In [7]:
channel_summary = (
    analysis.groupby("channel_title")
    .agg(
        videos=("video_id", "count"),
        median_views=("views", "median"),
        total_views=("views", "sum")
    )
    .sort_values("videos", ascending=False)
)

channel_summary.head(15)

,videos,median_views,total_views
channel_title,,,
VikatanTV,208,665398.0,151026885
SAB TV,206,276507.0,80287318
ETV Plus India,206,320455.5,95562261
etvteluguindia,205,209475.0,89450329
Flowers Comedy,202,592998.5,134339392
Study IQ education,202,52044.5,11422244
Tarang TV,199,85591.0,20019000
SET India,199,649790.0,191136460
Mazhavil Manorama,196,255161.0,62898862


In [8]:
print("Rows:", len(analysis))
print("Columns:", len(analysis.columns))
print("Duplicate rows:", analysis.duplicated().sum())

Rows: 33089
Columns: 24
Duplicate rows: 0


## Week 3 Observations

The dataset was transformed into a more analysis-ready structure.

Category metadata was combined with the video data, and additional variables such as total engagement, like rate, comment rate, and title length were created.

Grouped summaries were generated at both category and channel level to support the exploratory analysis performed in the following week.